In [ ]:
# parameters
# BINDER_FAST: set N=8, n_steps=40, n_iter_grape=50 for fast cloud execution
N = 15              # Hilbert space truncation (Fock dimension)
T_gate_ns = 500     # total gate time in nanoseconds
n_steps = 100       # number of time slices for the GRAPE propagator
n_iter_grape = 150  # maximum GRAPE iterations


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.gates import (
    snap_operator,
    snap_unitary_ideal,
    apply_snap,
    optimize_grape,
    optimize_crab,
    GRAPEResult,
)
from bosonic_gates.gates.snap import snap_phase_gradient
from bosonic_gates.gates.grape import _propagator_qutip
from bosonic_gates.states import fock_state, coherent_state


## Module 4c: Gate Benchmarking and Optimal Control

**Learning objectives:**
- Compute average gate fidelity via the Nielsen trace formula
- Understand how small pulse errors manifest as fidelity loss
- Run GRAPE optimal control to find a pulse that implements a target SNAP gate
- Compare GRAPE and CRAB strategies — convergence speed and final fidelity
- Compare ideal vs GRAPE-optimised cavity states via Wigner tomography
- Quantify leakage outside the computational subspace

---

**Sections:**
[1 Gate Fidelity](#sec1) ·
[2 GRAPE](#sec2) ·
[3 CRAB](#sec3) ·
[4 Wigner Comparison](#sec4) ·
[5 Leakage](#sec5) ·
[Exercises](#exercises)


<a id="sec1"></a>
## 1  Gate Fidelity — the Nielsen Formula

The standard measure of unitary gate quality is the **process fidelity**
(Nielsen & Chuang, Eq. 9.173):

$$F = \frac{|\mathrm{Tr}(U^\dagger U_{\rm target})|^2}{d^2},$$

where $d = N$ is the Hilbert space dimension.  Properties:
- $F = 1$ for a perfect implementation ($U = e^{i\phi}U_{\rm target}$,
  global phase does not affect $F$).
- $F = 0$ for a gate orthogonal to the target.
- $0 \le F \le 1$ always.

The related **infidelity** $1 - F$ is often used as the optimisation objective.


In [ ]:
def gate_fidelity(U_achieved: qt.Qobj, U_target: qt.Qobj) -> float:
    """Process fidelity F = |Tr(U†V)|² / d² (Nielsen formula)."""
    d = U_target.shape[0]
    overlap = (U_achieved.dag() * U_target).tr()
    return float(np.abs(overlap)**2 / d**2)


# Target: SNAP with theta[5] = pi/2
thetas_target = np.zeros(N)
thetas_target[5] = np.pi / 2
U_target = snap_unitary_ideal(N, thetas_target)

# Perfect implementation
F_perfect = gate_fidelity(U_target, U_target)
print(f"Fidelity of perfect gate:           F = {F_perfect:.8f}  (should be 1.0)")

# Small angle error
errors = np.array([0.0, 0.01, 0.05, 0.1, 0.3, 0.5, np.pi])
fidelities = []
for err in errors:
    thetas_err = thetas_target.copy()
    thetas_err[5] += err
    U_err = snap_operator(N, thetas_err)
    fidelities.append(gate_fidelity(U_err, U_target))

print("\nFidelity vs angle error on theta[5]:")
for err, F in zip(errors, fidelities):
    print(f"  delta_theta = {err:.3f} rad  →  F = {F:.6f}")


In [ ]:
# Plot F vs angle error
err_fine = np.linspace(0, np.pi, 200)
F_curve  = []
for err in err_fine:
    thetas_e = thetas_target.copy()
    thetas_e[5] += err
    F_curve.append(gate_fidelity(snap_operator(N, thetas_e), U_target))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(err_fine, F_curve, lw=2)
ax.scatter(errors, fidelities, color="crimson", zorder=5, label="sampled points")
ax.set_xlabel(r"Angle error $\delta\theta_5$ (rad)")
ax.set_ylabel(r"Gate fidelity $F$")
ax.set_title(r"SNAP fidelity vs phase error on $|5\rangle$")
ax.axhline(1.0, color="gray", ls="--", lw=1)
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print(f"At delta_theta = 0.1 rad, 1-F = {1 - gate_fidelity(snap_operator(N, thetas_target + np.array([0]*5+[0.1]+[0]*(N-6))), U_target):.4f}")


---
**Lab note.** *For a SNAP gate that touches only one Fock level, $F$ depends
only on the error in $\theta_k$; other levels contribute 1 to the trace and
do not reduce fidelity.  For a full SNAP with $N$ non-zero phases, errors on
all levels contribute and the infidelity budget is distributed across all
phase channels.*


<a id="sec2"></a>
## 2  GRAPE: Optimal Pulse for an Ideal SNAP

**GRAPE** (GRadient Ascent Pulse Engineering, Khaneja *et al.* 2005) finds a
piecewise-constant control pulse $u(t)$ that maximises the gate fidelity.  The
Hamiltonian model here is:

$$\hat{H}(t) = \underbrace{-\frac{K}{2}\hat{n}(\hat{n}-1)}_{\hat{H}_0\ (\text{Kerr drift})}
            + u(t)\underbrace{(\hat{a}+\hat{a}^\dagger)}_{\hat{H}_{\rm ctrl}\ (\text{cavity drive})}$$

where $K$ is the self-Kerr nonlinearity of the cavity (makes levels unequally
spaced, enabling frequency-selective control), and $u(t)$ is the in-phase
drive amplitude in units of the Kerr frequency.

The target is a SNAP gate with $\theta_3 = \pi$ and $\theta_7 = \pi/2$.


In [ ]:
# Build the system Hamiltonians
a   = qt.destroy(N)
adag = a.dag()
n_op = adag * a

K = 1.0   # Kerr coefficient (in natural units; T_gate is in seconds)
H0 = -K / 2 * n_op * (n_op - qt.qeye(N))   # Kerr drift
H_ctrl_x = [a + adag]                        # X-quadrature drive

# Target SNAP
thetas_grape = np.zeros(N)
thetas_grape[3] = np.pi
thetas_grape[7] = np.pi / 2
target_U_grape = snap_unitary_ideal(N, thetas_grape)

print("Target SNAP:", {n: thetas_grape[n] for n in range(N) if abs(thetas_grape[n]) > 1e-10})
print(f"Gate time: {T_gate_ns} ns")
print(f"GRAPE iterations: {n_iter_grape}")
print(f"Time steps: {n_steps}")


In [ ]:
# Run GRAPE optimisation
# Gate time in seconds (GRAPE uses SI or whatever units H is in;
# here H0 is dimensionless so T_gate must be matched accordingly)
T_gate = T_gate_ns * 1e-9  # seconds

result_grape = optimize_grape(
    target_U_grape,
    H0,
    H_ctrl_x,
    T=T_gate,
    n_steps=n_steps,
    n_iter=n_iter_grape,
    seed=42,
    use_jax=True,
)

print(result_grape)
print(f"Initial fidelity:  {result_grape.fidelity_history[0]:.6f}")
print(f"Final fidelity:    {result_grape.fidelity:.6f}")
print(f"Final infidelity:  {result_grape.infidelity:.6e}")
print(f"Converged:         {result_grape.converged}")
print(f"Method:            {result_grape.method}")


In [ ]:
# Plot convergence and optimised pulse
tlist = np.linspace(0, T_gate_ns, n_steps)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: fidelity history
axes[0].plot(result_grape.fidelity_history, lw=2)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Gate fidelity $F$")
axes[0].set_title("GRAPE convergence")
axes[0].set_ylim(-0.05, 1.05)
axes[0].axhline(1.0, color="gray", ls="--", lw=1)

# Right: optimised pulse shape
pulse_grape = result_grape.pulse_params[0]  # first (only) control channel
axes[1].step(tlist, pulse_grape, where="post", lw=1.5)
axes[1].set_xlabel("Time (ns)")
axes[1].set_ylabel("Pulse amplitude $u(t)$")
axes[1].set_title("Optimised GRAPE pulse")

plt.suptitle("GRAPE optimal control for target SNAP gate", fontsize=12)
plt.tight_layout()
plt.show()


---
**Lab note.** *GRAPE uses automatic differentiation (JAX backend if available,
otherwise finite-difference L-BFGS-B) to compute $\partial F/\partial u_k$
analytically.  The Cayley approximant used for each time-slice propagator
is unitary by construction, ensuring $F \le 1$ throughout optimisation.*


<a id="sec3"></a>
## 3  CRAB: Gradient-Free Alternative

**CRAB** (Chopped Random Basis, Caneva *et al.* 2011) parametrises the pulse
as a sum of random Fourier components and uses a gradient-free optimiser
(Nelder-Mead):

$$u(t) = \sum_{k=1}^{K}\bigl[a_k\cos(\omega_k t) + b_k\sin(\omega_k t)\bigr]$$

where the $\omega_k$ are randomly chosen near harmonic multiples of $2\pi/T$.

**When to prefer CRAB over GRAPE:**
- Short gate times or low Hilbert space dimension (gradient computation dominates)
- When JAX/autodiff is unavailable
- As a crosscheck to avoid local minima in GRAPE

**When to prefer GRAPE:**
- Long gates ($T \gg 1/K$), large $n_{\rm steps}$
- When $F > 0.99$ is required (CRAB plateau typically lower)


In [ ]:
# Run CRAB on the same target
result_crab = optimize_crab(
    target_U_grape,
    H0,
    H_ctrl_x,
    T=T_gate,
    n_basis=6,
    n_iter=100,
    seed=42,
)

print(result_crab)
print(f"CRAB final fidelity:  {result_crab.fidelity:.6f}")
print(f"GRAPE final fidelity: {result_grape.fidelity:.6f}")


In [ ]:
# Compare pulse shapes and convergence
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Convergence: GRAPE
axes[0, 0].plot(result_grape.fidelity_history, lw=2, color="steelblue")
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Fidelity")
axes[0, 0].set_title("GRAPE convergence")
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].axhline(1.0, color="gray", ls="--")

# Convergence: CRAB
axes[0, 1].plot(result_crab.fidelity_history, lw=2, color="darkorange")
axes[0, 1].set_xlabel("Function evaluations")
axes[0, 1].set_ylabel("Fidelity")
axes[0, 1].set_title("CRAB convergence")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].axhline(1.0, color="gray", ls="--")

# Pulse shapes
axes[1, 0].step(tlist, result_grape.pulse_params[0], where="post",
                lw=1.5, color="steelblue", label="GRAPE")
axes[1, 0].set_xlabel("Time (ns)")
axes[1, 0].set_ylabel("Pulse amplitude")
axes[1, 0].set_title("GRAPE pulse")
axes[1, 0].legend()

axes[1, 1].plot(tlist, result_crab.pulse_params[0], lw=1.5,
                color="darkorange", label="CRAB")
axes[1, 1].set_xlabel("Time (ns)")
axes[1, 1].set_ylabel("Pulse amplitude")
axes[1, 1].set_title("CRAB pulse (smooth Fourier shape)")
axes[1, 1].legend()

plt.suptitle("GRAPE vs CRAB: convergence and pulse shape comparison", fontsize=12)
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"  GRAPE: F = {result_grape.fidelity:.6f},  {len(result_grape.fidelity_history)} iterations")
print(f"  CRAB:  F = {result_crab.fidelity:.6f},  {len(result_crab.fidelity_history)} evaluations")


---
**Lab note.** *CRAB pulses are smooth (band-limited) by construction, making
them easier to upload to arbitrary waveform generators (AWGs) without
aliasing.  GRAPE pulses can be piecewise-constant or discontinuous; in
practice they are post-filtered by the AWG bandwidth, which can degrade
fidelity if not accounted for in the optimisation.*


<a id="sec4"></a>
## 4  Wigner Function Comparison: Ideal vs GRAPE

We propagate an initial $|0\rangle$ cavity state through (a) the ideal SNAP
and (b) the GRAPE-optimised pulse, then compare their Wigner functions.  Any
residual fidelity loss should manifest as a distortion, rotation, or extra
interference features in the Wigner function.


In [ ]:
# Initial state: coherent state (interesting Wigner)
alpha_init = 1.5
psi_init = qt.coherent(N, alpha_init)

# (a) Ideal SNAP
psi_ideal = target_U_grape * psi_init

# (b) GRAPE-optimised propagator
U_grape_np = _propagator_qutip(
    H0, H_ctrl_x, result_grape.pulse_params, T_gate, n_steps
)
psi_grape_out = qt.Qobj(U_grape_np @ psi_init.full())
psi_grape_out.dims = [[N], [1]]
psi_grape_out = psi_grape_out.unit()

# State fidelity
state_fid = abs(float(psi_ideal.dag() * psi_grape_out))**2
print(f"State fidelity (ideal vs GRAPE output): {state_fid:.6f}")
print(f"Process fidelity (from GRAPE result):   {result_grape.fidelity:.6f}")


In [ ]:
# Wigner functions side-by-side
xvec = np.linspace(-4.5, 4.5, 120)

W_ideal  = qt.wigner(qt.ket2dm(psi_ideal),     xvec, xvec)
W_grape  = qt.wigner(qt.ket2dm(psi_grape_out), xvec, xvec)
W_diff   = W_ideal - W_grape

vmax = max(np.max(np.abs(W_ideal)), np.max(np.abs(W_grape)))
vmax_diff = np.max(np.abs(W_diff))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, W, title, vm in zip(
    axes,
    [W_ideal, W_grape, W_diff],
    [r"(a) Ideal SNAP",
     rf"(b) GRAPE ($F={result_grape.fidelity:.4f}$)",
     r"(c) Difference (ideal − GRAPE)"],
    [vmax, vmax, vmax_diff],
):
    ax.contourf(xvec, xvec, W, levels=60, cmap="RdBu_r", vmin=-vm, vmax=vm)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"$X$")
    ax.set_ylabel(r"$P$")
    ax.set_aspect("equal")

plt.suptitle(rf"Wigner functions: $|\alpha_0| = {alpha_init}$ coherent state through SNAP gate",
             fontsize=11)
plt.tight_layout()
plt.show()

print(f"Max |W_diff| = {vmax_diff:.4f}  (0 = perfect match)")


**Interpretation.** Panel (c) shows the Wigner-function mismatch.  A non-zero
difference indicates where in phase space the GRAPE gate deviates from the
ideal.  Common patterns:
- *Rotation* — residual free-evolution (global phase, not a concern for $F$)
- *Amplitude mismatch* — insufficient pulse power
- *Extra fringes* — leakage to or from high-$n$ Fock levels

---
**Lab note.** *In experiment you perform Wigner tomography by measuring the
photon-number parity of the displaced state: $W(\alpha) = (2/\pi)
\mathrm{Tr}[\hat{D}^\dagger(\alpha)\,\hat{\Pi}\,\hat{D}(\alpha)\,\rho]$
where $\hat{\Pi} = e^{i\pi\hat{n}}$.  Each point requires $\sim 10^4$
repeated measurements.*


<a id="sec5"></a>
## 5  Leakage Outside the Computational Subspace

For a cavity qubit encoded in $\{|0\rangle, |1\rangle\}$, **leakage** is the
population that escapes into $|n \ge 2\rangle$:

$$L_1 = 1 - P_{\rm comp} = 1 - (\langle 0|\rho|0\rangle + \langle 1|\rho|1\rangle).$$

Even an ideal SNAP gate on a non-computational input state can scatter
population.  For a gate acting on the *full* Fock space the relevant concept
is instead the population that leaves the subspace where the gate was designed
to act (e.g., the truncated-$N$ space).  We illustrate both.


In [ ]:
# Leakage for a computational qubit (|0>, |1>) after the GRAPE pulse
# Start from |1>
psi_1 = qt.basis(N, 1)

U_grape_full = qt.Qobj(U_grape_np)
U_grape_full.dims = [[N], [N]]

psi_1_out = U_grape_full * psi_1
rho_1_out = qt.ket2dm(psi_1_out)

P0 = float(rho_1_out[0, 0].real)
P1 = float(rho_1_out[1, 1].real)
P_comp = P0 + P1
L1 = 1 - P_comp

print(f"After GRAPE SNAP, initial |1>:")
print(f"  P(0) = {P0:.6f}")
print(f"  P(1) = {P1:.6f}")
print(f"  P_comp = {P_comp:.6f}")
print(f"  Leakage L1 = {L1:.6f}")


In [ ]:
# Leakage vs gate time T (shorter T -> faster but more leakage from truncation)
T_scan_ns = np.array([50, 100, 200, 300, 500, 750, 1000])
leakage_scan = []
fidelity_scan = []

for T_ns in T_scan_ns:
    res = optimize_grape(
        target_U_grape, H0, H_ctrl_x,
        T=T_ns * 1e-9, n_steps=n_steps,
        n_iter=80, seed=0, use_jax=True,
    )
    fidelity_scan.append(res.fidelity)

    U_t = _propagator_qutip(H0, H_ctrl_x, res.pulse_params,
                            T_ns * 1e-9, n_steps)
    psi_out_t = qt.Qobj(U_t @ psi_1.full())
    psi_out_t.dims = [[N], [1]]
    rho_t = qt.ket2dm(psi_out_t)
    L = 1 - float(rho_t[0, 0].real) - float(rho_t[1, 1].real)
    leakage_scan.append(L)

print("Gate time vs fidelity and leakage:")
print(f"{'T (ns)':>8}  {'F':>10}  {'L1':>10}")
for T_ns, F, L in zip(T_scan_ns, fidelity_scan, leakage_scan):
    print(f"  {T_ns:6d}   {F:.6f}   {L:.6f}")


In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
color1, color2 = "steelblue", "crimson"

ax1.plot(T_scan_ns, fidelity_scan, "o-", color=color1, label="Gate fidelity $F$")
ax1.set_xlabel("Gate time $T$ (ns)")
ax1.set_ylabel("Gate fidelity $F$", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_ylim(0, 1.05)

ax2 = ax1.twinx()
ax2.plot(T_scan_ns, leakage_scan, "s--", color=color2, label="Leakage $L_1$")
ax2.set_ylabel("Leakage $L_1$", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

ax1.set_title("Fidelity and leakage vs gate time for GRAPE-optimised SNAP")
plt.tight_layout()
plt.show()


**Observation.** Longer gate times generally allow GRAPE to find higher-fidelity
solutions with lower leakage — the optimizer has more time (more time slices at
fixed $n_{\rm steps}$) to sculpt the pulse.  However, longer gates also spend
more time accumulating decoherence.  The optimal gate time balances these two
effects.

---
**Lab note.** *In experiment the gate-time trade-off is set by $T_1$, $T_2$,
and the Kerr nonlinearity: longer gates are more coherent but slower.  Typical
bosonic gate times are $100\,\text{ns}$–$1\,\mu\text{s}$ for
$K/2\pi \approx 1\,\text{MHz}$.  Leakage is suppressed by DRAG-like
corrections applied during post-processing of the optimal pulse.*


<a id="exercises"></a>
## Exercises

**Exercise 1.** Use GRAPE to find a pulse that maps $|0\rangle \to |1\rangle$
(a $\pi$ pulse on the cavity).  The target unitary is the permutation matrix
that swaps $|0\rangle$ and $|1\rangle$: build it as
`U_pi = qt.Qobj(np.eye(N)[[1,0]+list(range(2,N))])`.  Compute the fidelity
and plot the pulse.


In [ ]:
# Exercise 1
# YOUR CODE HERE


**Exercise 2.** Add a photon-loss collapse operator to the optimisation:
```python
kappa = 1e3   # Hz, T1 = 1/kappa = 1 ms
c_ops = [np.sqrt(kappa) * qt.destroy(N)]
```
Run `optimize_grape` with `c_ops=c_ops` and the same target.  Does the optimal
pulse change shape?  Does the fidelity drop?


In [ ]:
# Exercise 2
# YOUR CODE HERE


In [ ]:
# Solution — Exercise 1: pi pulse |0> -> |1>
perm = list(range(N))
perm[0], perm[1] = 1, 0
U_pi_target = qt.Qobj(np.eye(N)[perm, :])
U_pi_target.dims = [[N], [N]]

result_pi = optimize_grape(
    U_pi_target, H0, H_ctrl_x,
    T=T_gate, n_steps=n_steps,
    n_iter=n_iter_grape, seed=7, use_jax=True,
)

print(f"Pi-pulse fidelity: {result_pi.fidelity:.6f}")

# Verify state mapping
U_pi_np = _propagator_qutip(H0, H_ctrl_x, result_pi.pulse_params, T_gate, n_steps)
psi0_out = qt.Qobj(U_pi_np @ qt.basis(N, 0).full())
psi0_out.dims = [[N], [1]]
P_1 = abs(float(qt.basis(N, 1).dag() * psi0_out))**2
print(f"P(|1>) after pi pulse on |0>: {P_1:.6f}  (should be ~1)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(result_pi.fidelity_history, lw=2)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Fidelity")
axes[0].set_title("GRAPE convergence (pi pulse)")
axes[0].set_ylim(0, 1.05)

tlist_s = np.linspace(0, T_gate_ns, n_steps)
axes[1].step(tlist_s, result_pi.pulse_params[0], where="post", lw=1.5)
axes[1].set_xlabel("Time (ns)")
axes[1].set_ylabel("Pulse amplitude")
axes[1].set_title("Optimised pi-pulse shape")

plt.tight_layout()
plt.show()
